# LLM Ablation Analysis — Memory Turns & Prompt Style

This notebook explores how `memory_turns` and `prompt_style` affect LLM agent performance in the openspiel-arena tournament.

## Ablation Variables

| Variable | Values tested |
|----------|--------------|
| `memory_turns` | 0, 1 (baseline), 3 |
| `prompt_style` | `board_summary_then_choice` (baseline), `reason_then_choice` |

All runs use `tic_tac_toe`, 8 rounds per pairing, vs `random` and `mcts-50`.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Project imports
from scripts.plot_llm_ablation import (
    load_csvs,
    compute_metrics,
    compute_metrics_by_opponent,
    parse_memory_turns,
    parse_prompt_style,
)

plt.rcParams.update({"figure.dpi": 100, "figure.figsize": (10, 5)})

## 1. Load Data

In [ ]:
RESULTS_DIR = Path("results")
df = load_csvs([RESULTS_DIR])
print(f"Loaded {len(df)} matches from {RESULTS_DIR}")
df.head()

## 2. Identify LLM Variants

In [ ]:
# Filter to rows where at least one agent is an LLM variant
llm_mask = df["agent_a"].str.startswith("llm-") | df["agent_b"].str.startswith("llm-")
llm_df = df[llm_mask].copy()
print(f"{len(llm_df)} matches involve LLM agents")

# Show unique LLM agents
llm_agents = sorted(
    set(df["agent_a"]).union(df["agent_b"]) - {a for a in set(df["agent_a"]).union(df["agent_b"]) if not a.startswith("llm-")}
)
print(f"LLM variants: {llm_agents}")

## 3. Aggregate Metrics

In [ ]:
metrics = compute_metrics(df)
metrics_df = pd.DataFrame([
    {
        "variant": m.variant,
        "memory_turns": m.memory_turns,
        "prompt_style": m.prompt_style,
        "games": m.games,
        "win_rate": m.win_rate,
        "avg_invalid_retries": m.avg_invalid_retries,
        "avg_latency_ms": m.avg_latency_ms,
    }
    for m in metrics
])
metrics_df

## 4. Win Rate by Memory Turns (All Opponents)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

mem_groups = metrics_df.groupby("memory_turns")["win_rate"].mean().sort_index()

ax.bar(mem_groups.index.astype(str), mem_groups.values, color=["#e74c3c", "#2ecc71", "#3498db"])
ax.set_xlabel("Memory Turns")
ax.set_ylabel("Win Rate")
ax.set_title("Average Win Rate by Memory Turns (across all opponents)")
ax.set_ylim(0, 1)

for i, (idx, val) in enumerate(mem_groups.items()):
    ax.text(i, val + 0.02, f"{val:.1%}", ha="center", fontsize=12)

plt.tight_layout()
plt.show()

## 5. Win Rate Grouped by Opponent

In [ ]:
metrics_by_opp = compute_metrics_by_opponent(df)

# Build a table: variant × opponent → win_rate
rows = []
for opp, mlist in metrics_by_opp.items():
    for m in mlist:
        rows.append({"variant": m.variant, "opponent": opp, "win_rate": m.win_rate, "games": m.games})

opp_df = pd.DataFrame(rows)
pivot = opp_df.pivot_table(index="variant", columns="opponent", values="win_rate")

fig, ax = plt.subplots(figsize=(12, 5))
pivot.plot(kind="bar", ax=ax, width=0.7)
ax.set_ylabel("Win Rate")
ax.set_title("Win Rate by Variant and Opponent")
ax.set_ylim(0, 1.05)
ax.legend(title="Opponent")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 6. Invalid Move Retry Rate

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(metrics_df["variant"], metrics_df["avg_invalid_retries"], color="#e67e22")
ax.set_xlabel("LLM Variant")
ax.set_ylabel("Avg Invalid Retries")
ax.set_title("Invalid Move Retry Rate by Variant")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 7. Latency by Variant

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(metrics_df["variant"], metrics_df["avg_latency_ms"], color="#3498db")
ax.set_xlabel("LLM Variant")
ax.set_ylabel("Avg Latency (ms)")
ax.set_title("Average Latency by Variant")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 8. Key Findings

Summarise the ablation results here after running the tournaments.

In [ ]:
# Print summary table
print(metrics_df.to_string(index=False))